# Experiment runner (DINOv2 + adapters)

This notebook drives the **histopathology OOD patch classification** pipeline.  
All implementation lives under the importable package **`utils/`** so `DataLoader(num_workers>0)` works with multiprocessing `spawn` (macOS, Colab).

**Quick start:** run the first code cell, then edit **`RUNS`** and execute the launch cell.


# TO DO (research)

- Tissue / artefact QC (e.g. all-black or all-white patches)
- Stain augmentation and normalization (H&E literature)
- Alternative backbones: CTransPath, UNI, Virchow
- Adapters: AdaptFormer, LoRA / VeRA


## 0) Environment and imports


Add the repository root to `sys.path` (required in Jupyter / Colab when the package is not installed with `pip`).

The first cell imports configuration types, `run_experiment`, and prints `DEVICE`.


In [1]:
import sys
from pathlib import Path

# Repository root (run the notebook from the repo root, or adjust this path in Colab)
_ROOT = Path.cwd().resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import pandas as pd

from utils.constants import DEVICE, RUNS_DIR, TEST_IMAGES_PATH, TRAIN_IMAGES_PATH, VAL_IMAGES_PATH
from utils.common import ensure_dir, safe_json, seed_everything
from utils.config import (
    EarlyStoppingConfig,
    ModelConfig,
    ModuleSpec,
    ProcessingConfig,
    RunConfig,
    TrainConfig,
)
from utils.model import DefaultBinaryHead
from utils.outliers import MethodCOutlierParams
from utils.experiment import run_experiment

print(f"Using device: {DEVICE}")
print(f"H5 paths (override in utils/constants.py if needed): train={TRAIN_IMAGES_PATH!r}")


Using device: cpu
H5 paths (override in utils/constants.py if needed): train='train.h5'


### Method C outlier detection (`ExtractOutlier`)

Implementation: **`utils/outliers.py`** (same rules as `outlier_detection.ipynb` on **raw H5** resolution).

- **Train:** flagged patch IDs are **removed** from training.
- **Val / test:** outliers stay in the loader; predictions are **0** without a forward pass.

Parallelism for the scan uses **threads** (`ThreadPoolExecutor`), controlled by `outlier_scan_workers` in `RunConfig`.


## 1) Experiment configuration


Edit the **`RUNS`** list in the last section. Types are dataclasses in **`utils/config.py`**:

- `ProcessingConfig` — resize and optional numpy / sklearn steps  
- `ModelConfig` — DINO backbone name, `ModuleSpec` for adapter and head  
- `TrainConfig` — batch size, LR, `num_workers` (PyTorch DataLoader), early stopping  
- `RunConfig` — run name, seeds, optional `MethodCOutlierParams`, `data_fraction` smoke tests, test prediction  

`ModuleSpec(enabled=False)` for the adapter triggers **one** frozen-backbone pass and **head-only** training on embeddings (linear probing).


(No code cell here — imports are in section 0.)


## 2) Data pipeline


**Module:** `utils/data.py`

- `H5BinaryDataset` — HDF5 patches, picklable for multiprocessing  
- `PreprocessingTransform` / `build_preprocessing` — picklable transforms  
- `EmbeddingTensorDataset` / `EmbeddingValDataset` — precomputed DINO features  
- `CenterProportionalBatchSampler` — optional center-balanced batches


## 3) Model


**Module:** `utils/model.py`

- `load_frozen_backbone` — DINOv2 from `torch.hub`  
- `DefaultMLPAdapter`, `DefaultBinaryHead`  
- `FullModel` — backbone + optional adapter + head  
- `HeadOnly` — linear probe on fixed embeddings


## 4) Training loop and metrics


**Module:** `utils/training.py`

`Trainer` performs train/val epochs, per-center metrics, CSV logging (`metrics.csv`), ROC/PR curve `.npz` files, early stopping, `best.pt` / `last.pt`.


## 5) Test prediction


**Module:** `utils/predict.py` — `predict_test(...)` writes `predictions.csv` when `do_predict_test=True` in `RunConfig`.


## 6) Experiment runner


**Module:** `utils/experiment.py` — `run_experiment(run_cfg)` orchestrates outlier filtering, DINO precompute (if adapter disabled), optimization, and optional test inference.

Outputs under `runs/<run_name>/seed_<k>/`: `config.json`, `metrics.csv`, `checkpoints/`, `curves/`.


## 7) Define `RUNS` and launch

Two baselines (toggle comments to switch):

1. **Baseline** — no outlier filter, full data.  
2. **Baseline + Method C** — same model, with `MethodCOutlierParams` and optional `data_fraction` for quick tests.

Both use **linear probing** (`adapter.enabled=False`): one backbone precompute, then head-only epochs.


### Summary of the two runs below

| Run | Outliers | Notes |
|-----|----------|--------|
| `baseline_no_outlier_filter` | Off | Full train/val |
| `baseline_method_c_outliers` | Method C | Same hyperparameters + outlier scan |

Both entries run sequentially; remove or comment any run you do not need.


AJOUTER VAL PREDICTION POUR CHECKER LES OUTLIRS

In [4]:
# Shared linear-probing setup: DINO frozen, single precompute, train head only.
_SHARED_MODEL = ModelConfig(
    backbone_name="dinov2_vits14",
    adapter=ModuleSpec(enabled=False, module_cls=None, module_kwargs=None),
    head=ModuleSpec(
        enabled=True,
        module_cls=DefaultBinaryHead,
        module_kwargs={},
    ),
)

center_proportions = {
    0: 17756 / 100_000,
    3: 38756 / 100_000,
    4: 43488 / 100_000,
}

_SHARED_TRAIN = TrainConfig(
    num_workers=8,
    batch_size=16,
    lr=1e-3,
    num_epochs=100,
    early_stopping=EarlyStoppingConfig(monitor="val_loss", mode="min", patience=10),
    use_center_balanced_batches=True,
    center_proportions=center_proportions,
)

_BASELINE_TRAIN = TrainConfig(
    num_workers=8,
    batch_size=16,
    lr=1e-3,
    num_epochs=100,
    early_stopping=EarlyStoppingConfig(monitor="val_loss", mode="min", patience=10),
    use_center_balanced_batches=False,
)

RUNS: list[RunConfig] = [
    # 1) Baseline: no outlier scan
    # RunConfig(
    #     run_name="baseline",
    #     seeds=[0],
    #     processing=ProcessingConfig(resize_hw=(98, 98)),
    #     model=_SHARED_MODEL,
    #     train=_BASELINE_TRAIN,
    #     outlier_params=None,
    #     data_fraction=None,
    #     do_predict_test=True,
    # ),
    RunConfig(
        run_name="baseline_balanced",
        seeds=[0],
        processing=ProcessingConfig(resize_hw=(98, 98)),
        model=_SHARED_MODEL,
        train=_SHARED_TRAIN,
        outlier_params=None,
        data_fraction=None,
        do_predict_test=True,
    ),
    # 2) Baseline + Method C outlier filter (optional smoke-test fraction + test prediction)
    # RunConfig(
    #     run_name="baseline_outliers6",
    #     seeds=[0],
    #     processing=ProcessingConfig(resize_hw=(98, 98)),
    #     model=_SHARED_MODEL,
    #     train=_BASELINE_TRAIN,
    #     data_fraction=None,
    #     outlier_params=MethodCOutlierParams(
    #         min_sat_mean=0.03,
    #         min_colorfulness=0.03,
    #         min_grad_energy=0.005,
    #         white_gray_min=0.86,
    #         white_max_sat=0.28,
    #         max_white_frac=0.6,
    #     ),
    #     outlier_scan_workers=8,
    #     outlier_scan_batch_size=1000,
    #     do_predict_test=True,
    # ),
]

results = []
for rc in RUNS:
    print(f"\n=== Starting run: {rc.run_name} ===")
    out = run_experiment(rc)
    results.append(out)

rows = []
for r in results:
    for sr in r["seed_results"]:
        rows.append(
            {
                "run_name": r["run_name"],
                "seed": sr["seed"],
                "best_epoch": sr.get("best_epoch"),
                "monitor": sr.get("early_stopping_monitor"),
                "best_monitor_value": sr.get("best_monitor_value"),
                "time_sec": sr.get("time_sec"),
            }
        )

pd.DataFrame(rows).sort_values(["run_name", "seed"])



=== Starting run: baseline_balanced ===


Using cache found in /Users/user/.cache/torch/hub/facebookresearch_dinov2_main


precompute train seed=0:   0%|          | 0/6250 [00:17<?, ?it/s]

precompute val seed=0:   0%|          | 0/2182 [00:25<?, ?it/s]

Using cache found in /Users/user/.cache/torch/hub/facebookresearch_dinov2_main


...

train:   0%|          | 0/5918 [00:00<?, ?it/s]

eval:   0%|          | 0/2182 [00:00<?, ?it/s]

/Users/user/Desktop/GitHub/mva-dlmi-2026-histopathology-ood-classification/utils/experiment.py:340: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(os.path.j

predict:   0%|          | 0/85054 [00:17<?, ?it/s]

,run_name,seed,best_epoch,monitor,best_monitor_value,time_sec
0,baseline_balanced,0,12,val_loss,0.302777,121.7513
